# D4 — The walk operator **is** attention / El operador de walk **es** atención

🇬🇧 This is the central idea of the whole project. We build it in three small steps:
1. Recall how a Transformer's **attention** works (one line).
2. Notice that aggregating over length-`g` walks has the *exact same shape*.
3. Conclude: the path algebra gives us a **sparse, multi-hop attention** — and that is what beats over-squashing.

🇪🇸 Esta es la idea central de todo el proyecto. La construimos en tres pasos pequeños:
1. Recordar cómo funciona la **atención** de un Transformer (una línea).
2. Notar que agregar sobre recorridos de longitud `g` tiene *exactamente la misma forma*.
3. Concluir: el álgebra de caminos nos da una **atención sparse y multi-salto** — y eso vence al over-squashing.

## Step 1 — attention in one line / Paso 1 — atención en una línea

🇬🇧 A Transformer updates token `j` by a **weighted sum** of all other tokens' values:
$$h_j = \sum_i \alpha_{ji}\, v_i, \qquad \alpha_{ji} = \text{softmax}_i\big(q_j \cdot k_i\big).$$
The weights `α` are **learned** from content (query · key). Standard attention is **all-pairs**: every token
may attend to every other.

🇪🇸 Un Transformer actualiza el token `j` con una **suma ponderada** de los valores de los demás:
$$h_j = \sum_i \alpha_{ji}\, v_i, \qquad \alpha_{ji} = \text{softmax}_i\big(q_j \cdot k_i\big).$$
Los pesos `α` se **aprenden** del contenido (query · key). La atención estándar es **todos-contra-todos**.

## Step 2 — the same shape, but over walks / Paso 2 — la misma forma, pero sobre recorridos

🇬🇧 Now look at how a node aggregates information from `g` hops away. It is also a weighted sum over nodes:
$$h_j = \sum_i W^g_{ji}\, (\text{message from } i).$$
The only questions are: **(a) which `i` are included**, and **(b) what are the weights `W`**. The path algebra
answers (a): node `i` is included exactly when there is a length-`g` walk `i → j` — i.e. when `n_g(i,j) > 0`.
That is a **mask**.

🇪🇸 Mira cómo un nodo agrega información de `g` saltos. También es una suma ponderada sobre nodos:
$$h_j = \sum_i W^g_{ji}\, (\text{mensaje de } i).$$
Las preguntas son: **(a) qué `i` se incluyen**, y **(b) cuáles son los pesos `W`**. El álgebra responde (a):
`i` se incluye exactamente cuando hay un recorrido de longitud `g` `i → j` — cuando `n_g(i,j) > 0`. Una **máscara**.

<img src="img/03_dense_vs_sparse_attention.svg" width="720" alt="Dense all-pairs attention fills the whole matrix; walk-masked attention keeps only the reachable pairs (about 2%)."/>

<sub>🇬🇧 Standard attention scores every pair (left). The path algebra keeps only walk-reachable pairs (right) — global reach, ~2% of the cost. · 🇪🇸 La atención estándar puntúa todo par (izq.). El álgebra conserva solo los pares alcanzables (der.) — alcance global, ~2% del costo.</sub>

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators

data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
ei, N = data.edge_index.numpy(), data.num_nodes
raw, _ = walk_operators(ei, N, max_length=4)

# the attention SUPPORT at range g = where walks exist / el SOPORTE de atención en rango g
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for g in range(4):
    mask = (raw[g] > 0).astype(int)
    axes[g].imshow(mask, cmap='Greens', vmin=0, vmax=1)
    axes[g].set_title(f'range g={g+1}\n{mask.sum()} / {N*N} pairs')
    axes[g].set_xlabel('source i'); axes[g].set_ylabel('target j')
plt.suptitle('the path-reachability MASK = the attention support (sparse!)')
plt.tight_layout(); plt.show()

🇬🇧 Notice how **sparse** these masks are — mostly white. Standard attention would fill the whole square. The
path algebra tells us we only need the green entries — at the deepest range, about **2%** of the full square.
Global reach (multi-hop) at a tiny fraction of the cost.

🇪🇸 Fíjate qué **dispersas** son estas máscaras — casi todo blanco. La atención estándar llenaría el cuadrado.
El álgebra dice que solo necesitamos las entradas verdes — en el rango más profundo, ~**2%** del cuadrado.
Alcance global (multi-salto) a una fracción mínima del costo.

## Step 3 — fixed weights vs. learned weights / Paso 3 — pesos fijos vs. pesos aprendidos

🇬🇧 For (b), the weights, there are two choices:
- **Fixed** `W = n_g(i,j)` (the raw path count). This is `walkraw`. It reaches the target, but weights *every*
  reachable source the same — it cannot say *which* one matters.
- **Learned** `W = α_{ji}` from query·key, over the masked pairs only. This is `WalkAttention` — exactly
  Transformer attention restricted to walk-reachable pairs.

🇪🇸 Para (b), los pesos, hay dos opciones:
- **Fijos** `W = n_g(i,j)` (conteo crudo). Es `walkraw`. Alcanza, pero pesa *todas* las fuentes igual — no
  puede decir *cuál* importa.
- **Aprendidos** `W = α_{ji}` de query·key, solo sobre los pares de la máscara. Es `WalkAttention` — atención
  de Transformer restringida a los pares alcanzables.

<img src="img/04_fixed_vs_learned.svg" width="720" alt="Fixed weights treat every reachable source equally; learned attention focuses on the source that holds the answer."/>

<sub>🇬🇧 Fixed path-count weights are all equal (left) — `walkraw`. Learned attention concentrates on the relevant source (right) — `WalkAttention`. · 🇪🇸 Los pesos por conteo son todos iguales (izq.) — `walkraw`. La atención aprendida se concentra en la fuente relevante (der.) — `WalkAttention`.</sub>

In [ ]:
from oversquash.attention import WalkAttention
from torch_geometric.utils import softmax as pyg_softmax
g = 3  # deepest range / rango más profundo
t = int(data.root_mask.nonzero()[0])

# Build the mask the SAME way the real layer does: transpose so indices()[0]=target, [1]=source
# (mirrors WalkAttention.forward exactly). / Como la capa real: transpuesta, [0]=objetivo, [1]=fuente.
mask = torch.from_numpy((raw[g] > 0).T.astype('float32')).to_sparse().coalesce()
tgt, src = mask.indices()[0], mask.indices()[1]

# FIXED weights = raw path counts (what walkraw uses) / pesos FIJOS = conteos crudos
Wfixed = raw[g][src.numpy(), tgt.numpy()]

# LEARNED weights from an (untrained) WalkAttention layer, to show they DIFFER per source
torch.manual_seed(0)
layer = WalkAttention(data.x.size(1), 8, n_heads=1)
q = layer.q(data.x).view(N, 1, 8); k = layer.k(data.x).view(N, 1, 8)
logit = (q[tgt] * k[src]).sum(-1) * layer.scale
alpha = pyg_softmax(logit, tgt, num_nodes=N).detach().numpy().ravel()

into_t = (tgt.numpy() == t)
print(f'Sources that reach target node {t}: {src.numpy()[into_t]}')
print(f'  FIXED   (walkraw):   {Wfixed[into_t].round(2)}   <- ALL EQUAL: cannot select')
print(f'  LEARNED (attention): {alpha[into_t].round(3)}   <- DIFFERENT per source: can select')

## The one-sentence summary / El resumen en una frase

🇬🇧 **The path algebra says WHO can attend to whom across many hops (a sparse mask); attention learns HOW MUCH
each one matters.** Fixed path-counts can't tell sources apart; learned attention can. That single difference
turns a partial fix into a complete one — which we prove with real training in **D5**.

🇪🇸 **El álgebra de caminos dice QUIÉN puede atender a quién a través de muchos saltos (una máscara sparse); la
atención aprende CUÁNTO importa cada uno.** Los conteos fijos no distinguen fuentes; la atención aprendida sí.
Esa única diferencia convierte un arreglo parcial en uno completo — lo demostramos con entrenamiento real en **D5**.